In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system:
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test the local server, run:
   ```bash
   curl http://localhost:11434
   ```

You should get a response like:
```json
{"models": [...]}
```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```


In [7]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I don’t see anything about “Olama” in the FAQ context.

If you mean running the course locally, the FAQ says you can do that if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. Codespaces is just the easiest way to start with the same environment.

If you meant a specific local model tool, please share the exact name and I can check the FAQ wording.


In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

answer = response.output_text
print(answer)

Yes—probably, as long as enrollment is still open.

A few things to check:
- **Start date / deadlines**: If the course hasn’t started yet, you can usually still join.
- **Enrollment policy**: Some courses allow late registration, while others require pre-enrollment.
- **Prerequisites**: Make sure you meet any required background or prior courses.
- **Capacity**: If the course is full, you may need to join a waitlist.

If you want, I can help you draft a short message to the instructor or course office asking whether you can still enroll.


In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment join"}', call_id='call_TNNnCykYPSDLtfb3Lz0R10wa', name='search', type='function_call', id='fc_0b74fa83ad898aac006a34d775b9c48198a20250c02a5814f0', namespace=None, status='completed')]

In [16]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

answer = response.output_text
print(answer)

Yes, you can still join and start learning at any time.

If you want a certificate, make sure you submit your project while submissions are still open.


In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(777, 35)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
